# Ringdown: frequency and damping from a struck record

Runs [`ringdown.py`](ringdown.py) over one window of one `campod-accel` capture and presents what
it found: the impulses it located, the bands it fitted, the modes that survived, and figures for
each. **It makes no argument about a particular airframe** -- what any of it means belongs in
`image-quality-experiments.md`, alongside the other evidence for whatever claim it bears on.

Point it at a capture and a time window. The window is required in practice because a capture
normally contains more than the strikes -- container startup, handling, pose changes -- all of
which are impulses that this would otherwise try to fit as modes.


In [ ]:
# --- parameters (papermill) ---
# Directory holding accel-*.jsonl for one pod session.
capture_dir = ("/home/jovyan/datasets/flights/260926-sixpose-and-bump/extracted/"
               "campod-sw/de59ef4b-37d4-41f2-b860-9532c76e233c")
# Seconds from the first sample of the capture. Everything below is restricted to this.
window_s = (283.0, 303.0)
# Which signal to fit. "mag" is rotation-independent, which matters when the sensor's
# orientation in the airframe is not established.
axis = "mag"
# Cap on how many candidate bands to fit. None = no cap, which is the default and the right
# one: prominence decides, and a count would drop a peak for being less prominent than five
# others, which says nothing about the peak. Set it only as a safety valve, and check below
# whether it bound -- a cap of 6 was silently excluding a mode fitted on multiple impulses.
n_bands = None
fmin_hz = 20.0
# Where ringdown.py lives, relative to the working directory the notebook is executed in.
analysis_dir = "analysis"
out_dir = None

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

ANALYSIS = Path(analysis_dir)
if not (ANALYSIS / "ringdown.py").exists():
    raise FileNotFoundError(
        f"no ringdown.py under {ANALYSIS.resolve()} -- set the `analysis_dir` parameter to the "
        "directory containing it, relative to where this notebook is being executed")
import sys
sys.path.insert(0, str(ANALYSIS))
import ringdown as rd

CAPTURE = Path(capture_dir)
OUT = Path(out_dir) if out_dir else CAPTURE
W0, W1 = window_s
COLOURS = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100")
GRID = dict(color="#d9d8d2", lw=0.6, alpha=0.9)

channels = {}
for path in sorted(CAPTURE.glob("accel-*.jsonl")):
    name = path.stem.replace("accel-", "")
    channels[name] = rd.load_accel_jsonl(path).window(W0, W1)
if not channels:
    raise FileNotFoundError(f"no accel-*.jsonl under {CAPTURE}")
print(f"{len(channels)} channel(s) over {W0:.1f}-{W1:.1f} s of {CAPTURE.name}")

## 1. What is in the window

Per channel, before fitting anything: how much data, at what fitted rate, whether any sample is
clipped, and whether the reader emitted samples continuously across it.


In [ ]:
window = {}
for name, cap in channels.items():
    gaps = cap.read_gaps()
    rates = cap.fit_rate_over(segment_s=max(1.0, (W1 - W0) / 6))
    spread_ppm = float((rates.max() - rates.min()) / rates.mean() * 1e6) if len(rates) > 1 else None
    window[name] = dict(
        n_samples=int(len(cap.time_s)),
        rate_hz=cap.rate_hz,
        rate_segment_spread_ppm=spread_ppm,
        saturated_samples=int(cap.saturated.sum()),
        range_g=cap.header.get("range_g"),
        peak_g=float(cap.magnitude.max()),
        read_stalls=gaps["n_stalls"],
        stall_time_ms=gaps["stall_time_s"] * 1e3,
        fifo_fill_ms=gaps["fill_s"] * 1e3,
        worst_read_gap_ms=gaps["worst_gap_s"] * 1e3,
    )
    w = window[name]
    print(f"{name:8s} {w['n_samples']:>8,d} samples at {w['rate_hz']:7.1f} Hz "
          f"(segment spread {w['rate_segment_spread_ppm']:.0f} ppm)" if spread_ppm is not None
          else f"{name:8s} {w['n_samples']:>8,d} samples at {w['rate_hz']:7.1f} Hz")
    print(f"         peak |a| {w['peak_g']:.1f} g against +/-{w['range_g']} g per axis; "
          f"{w['saturated_samples']} clipped samples")
    print(f"         {w['read_stalls']} read stall(s), {w['stall_time_ms']:.0f} ms unsampled; "
          f"worst read gap {w['worst_read_gap_ms']:.2f} ms against a "
          f"{w['fifo_fill_ms']:.2f} ms FIFO fill")

In [ ]:
# A clipped sample's true value is unknown, so a peak containing one is a lower bound and its
# flat top puts energy at harmonics the structure does not have. Reported, not corrected.
for name, w in window.items():
    if w["saturated_samples"]:
        print(f"{name}: {w['saturated_samples']} clipped samples in this window. Peak amplitudes "
              f"are lower bounds and the first millisecond of each impulse is not trustworthy.")
    if w["read_stalls"]:
        print(f"{name}: {w['read_stalls']} read stall(s) inside the window -- a decay fitted "
              f"across one is not a decay.")

## 2. Impulses located

Peak-picking with suppression, as a fraction of each channel's own largest excursion. The count
should match the number of impulses actually delivered; if it does not, `min_peak_frac` or the
window is wrong and nothing below is meaningful.


In [ ]:
taps = {}
for name, cap in channels.items():
    sig = dict(x=cap.x, y=cap.y, z=cap.z, mag=cap.magnitude)[axis]
    sig = sig - np.median(sig)
    check = rd.looks_like_a_bump_test(sig)
    found = rd.find_taps(cap.time_s, sig)
    taps[name] = dict(signal=sig, taps=found, check=check)
    t0 = cap.time_s[0]
    print(f"{name:8s} {len(found):3d} impulses; peak/median {check['peak_over_median']:.0f} "
          f"(needs >= {check['threshold']:.0f}), crest {check['crest']:.1f}, "
          f"impulses-in-silence: {check['ok']}")
    print(f"         at s: {', '.join(f'{cap.time_s[i]-t0:.1f}' for _, i in found)}")
counts = {n: len(v["taps"]) for n, v in taps.items()}
if len(set(counts.values())) > 1:
    print(f"\nCHANNELS DISAGREE on impulse count: {counts}. They saw the same physical events, "
          "so any per-mode comparison between them is comparing different fits.")

In [ ]:
fig, axes = plt.subplots(len(channels), 1, figsize=(14, 2.4 * len(channels)), sharex=True,
                         squeeze=False)
for ax, (name, cap) in zip(axes[:, 0], channels.items()):
    t0 = cap.time_s[0]
    sig = taps[name]["signal"]
    step = max(1, len(sig) // 20000)
    ax.plot(cap.time_s[::step] - t0, sig[::step], lw=0.4, color=COLOURS[0])
    for _, i in taps[name]["taps"]:
        ax.axvline(cap.time_s[i] - t0, color=COLOURS[1], lw=0.8, alpha=0.6)
    if cap.saturated.any():
        ax.plot(cap.time_s[cap.saturated] - t0, sig[cap.saturated], ".", ms=3,
                color="#c0392b", label="clipped")
        ax.legend(loc="upper right", fontsize=8)
    ax.set_ylabel(f"{name}\n{axis} (g)")
    ax.grid(**GRID)
axes[-1, 0].set_xlabel(f"seconds from {W0:.0f} s into the capture")
axes[0, 0].set_title(f"{CAPTURE.name}: window {W0:.0f}-{W1:.0f} s, impulses marked")
fig.tight_layout(); fig.savefig(OUT / "ringdown-impulses.png", dpi=130); plt.show()

## 3. Bands searched, and modes fitted

Each impulse is fitted separately in each band. The scatter across impulses is the error bar;
a mode seen on one impulse is not a mode.


In [ ]:
results, summaries, bands_used = {}, {}, {}
for name, cap in channels.items():
    sig = taps[name]["signal"]
    bands = rd.candidate_bands(cap.time_s, sig, fmin=fmin_hz, fmax=cap.rate_hz / 2.5, n=n_bands)
    bands_used[name] = [[float(lo), float(hi)] for lo, hi in bands]
    capped = n_bands is not None and len(bands) >= n_bands
    print(f"\n{name}: {len(bands)} band(s) cleared the prominence floor"
          + (f" (cap n_bands={n_bands})" if n_bands is not None else " (no cap)"))
    if capped:
        print(f"  the cap BOUND -- peaks below the {n_bands}th most prominent were not fitted")
    print("  " + ", ".join(f"{lo:.0f}-{hi:.0f}" for lo, hi in bands) + " Hz")
    if not bands:
        results[name], summaries[name] = {}, {}
        continue
    res = rd.analyze(cap.time_s, sig, bands=bands)
    results[name] = res
    summaries[name] = rd.summarize(res)
    modes = summaries[name].get("modes", [])
    print(f"  {res['n_taps']} impulses fitted -> {len(modes)} mode(s) above the r2 floor")
    for m in modes:
        sd = m.get("f_sd_hz")
        print(f"    {m['f_hz']:8.2f} Hz" + (f" +/- {sd:.2f}" if sd is not None else "         ")
              + f"   zeta {m['zeta']:.4f}   Q {m['q']:5.1f}"
              + f"   decay {m['decay_s']*1e3:6.1f} ms   r2 {m['r2_median']:.3f}"
              + f"   {m['n_taps']} impulse(s)")

In [ ]:
from scipy.signal import welch
fig, ax = plt.subplots(figsize=(14, 5))
for k, (name, cap) in enumerate(channels.items()):
    f, P = welch(taps[name]["signal"], fs=cap.rate_hz, nperseg=4096)
    c = COLOURS[k % len(COLOURS)]
    ax.semilogy(f, P, lw=0.8, color=c, label=f"{name} ({cap.rate_hz:.0f} Hz)")
    for m in summaries.get(name, {}).get("modes", []):
        multi = (m.get("n_taps") or 0) > 1
        ax.axvline(m["f_hz"], color=c, ls="--" if multi else ":",
                   lw=1.1 if multi else 0.7, alpha=0.8 if multi else 0.4)
ax.set_xlim(0, max(c.rate_hz for c in channels.values()) / 2.5)
ax.set_xlabel("Hz"); ax.set_ylabel(f"PSD of {axis} (g^2/Hz)")
ax.set_title("Window spectrum. Dashed = mode fitted on more than one impulse, dotted = one only.")
ax.grid(**GRID); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "ringdown-spectrum.png", dpi=130); plt.show()

In [ ]:
# The loudest impulse on each channel, close up: what the fit is fitting.
fig, axes = plt.subplots(1, len(channels), figsize=(7 * len(channels), 3.4), squeeze=False)
for ax, (name, cap) in zip(axes[0], channels.items()):
    sig = taps[name]["signal"]
    if not taps[name]["taps"]:
        ax.set_axis_off(); continue
    i = max((i for _, i in taps[name]["taps"]), key=lambda j: abs(sig[j]))
    t0 = cap.time_s[i]
    m = (cap.time_s >= t0 - 0.005) & (cap.time_s <= t0 + 0.20)
    ax.plot((cap.time_s[m] - t0) * 1e3, sig[m], lw=0.7, color=COLOURS[0])
    if cap.saturated[m].any():
        ax.plot((cap.time_s[m][cap.saturated[m]] - t0) * 1e3, sig[m][cap.saturated[m]],
                ".", ms=4, color="#c0392b", label="clipped")
        ax.legend(fontsize=8)
    ax.set_title(f"{name}: loudest impulse"); ax.set_xlabel("ms"); ax.set_ylabel(f"{axis} (g)")
    ax.grid(**GRID)
fig.tight_layout(); fig.savefig(OUT / "ringdown-one-impulse.png", dpi=130); plt.show()

## 4. Summary

Written to `ringdown.json` beside the capture. Numbers only -- the interpretation belongs with
the claim it supports, not here.


In [ ]:
summary = dict(
    capture=str(CAPTURE),
    session=next(iter(channels.values())).header.get("session"),
    node=next(iter(channels.values())).header.get("node"),
    window_s=[W0, W1],
    axis=axis,
    band_search=dict(fmin_hz=fmin_hz, n_bands_requested=n_bands,
                     n_bands_found={k: len(v) for k, v in bands_used.items()},
                     bands_hz=bands_used),
    per_channel={
        name: dict(
            window[name],
            impulses=len(taps[name]["taps"]),
            impulse_times_s=[float(channels[name].time_s[i] - channels[name].time_s[0])
                             for _, i in taps[name]["taps"]],
            impulses_in_silence=taps[name]["check"],
            modes=summaries.get(name, {}).get("modes", []),
        ) for name in channels
    },
)
if len(channels) == 2:
    a, b = list(channels)
    try:
        summary["paired_modes"] = rd.compare_arms(summaries.get(a, {}), summaries.get(b, {}))
    except Exception as exc:
        summary["paired_modes"] = {"error": str(exc)}
p = OUT / "ringdown.json"
p.write_text(json.dumps(summary, indent=1, default=float))
print(f"wrote {p}")
print(json.dumps(summary, indent=1, default=float)[:1800])